# Gemma SFT experiment: Colab fine-tuning

Fine-tunes the pretrained student (`baseline_full`) twice on the same prompts, once with the human Dolly answers and once with the Gemma 4 answers, then compares them. Set the runtime to **T4 GPU** first.

Run the cells **one at a time, in order**. Each fine-tuning run takes a few minutes. Do not use "Run all".

Before starting, `tensorix/v2/sft/` on Drive must contain `human_train.npz`, `gemma_train.npz`, `human_eval.npz`, `gemma_eval.npz` and `prompts.jsonl` (made by `04_tokenize_sft.py` and `02_make_prompts.py`).

In [ ]:
# CELL 1: SETUP (run first in every new session)
import os, torch
BRANCH = "feature/gemma-sft"      # change to "main" once the experiment is merged
REPO   = "https://github.com/sunidhishekar-create/tensorix-kd-slm.git"
DRIVE  = "/content/drive/MyDrive/tensorix/v2"

from google.colab import drive
drive.mount("/content/drive")

%cd /content
if not os.path.exists("/content/tensorix-kd-slm"):
    !git clone {REPO}
%cd /content/tensorix-kd-slm
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
!pip install -q -r requirements.txt

os.makedirs("/content/sft", exist_ok=True)
!cp {DRIVE}/sft/*.npz {DRIVE}/sft/prompts.jsonl /content/sft/
!ls -lh /content/sft

INIT = f"{DRIVE}/checkpoints/baseline_full/best.pt"
need = ["human_train.npz", "gemma_train.npz", "human_eval.npz", "gemma_eval.npz", "prompts.jsonl"]
missing = [f for f in need if not os.path.exists(f"/content/sft/{f}")]
assert not missing, f"missing in /content/sft: {missing}. Copy them to {DRIVE}/sft first."
assert os.path.exists(INIT), f"pretrained checkpoint not found: {INIT}"
assert torch.cuda.is_available(), "No GPU. Runtime, Change runtime type, T4 GPU, then rerun Cell 1."
print("GPU:", torch.cuda.get_device_name(0), "| setup OK")

In [ ]:
# CELL 2: FINE-TUNE ON HUMAN ANSWERS (Model A)
%cd /content/tensorix-kd-slm
OUT_H = f"{DRIVE}/sft/sft_human"
!python experiments/gemma_sft/05_finetune.py --train /content/sft/human_train.npz --eval /content/sft/human_eval.npz /content/sft/gemma_eval.npz --init {INIT} --out_dir {OUT_H} --prompts /content/sft/prompts.jsonl

In [ ]:
# CELL 3: FINE-TUNE ON GEMMA ANSWERS (Model B). Identical settings; only the training file differs.
%cd /content/tensorix-kd-slm
OUT_G = f"{DRIVE}/sft/sft_gemma"
!python experiments/gemma_sft/05_finetune.py --train /content/sft/gemma_train.npz --eval /content/sft/human_eval.npz /content/sft/gemma_eval.npz --init {INIT} --out_dir {OUT_G} --prompts /content/sft/prompts.jsonl

In [ ]:
# CELL 4: COMPARE. Answer-token loss on held-out prompts, before fine-tuning (epoch 0) and after each epoch.
import pandas as pd
for name, out in (("A: trained on HUMAN answers", OUT_H), ("B: trained on GEMMA answers", OUT_G)):
    print(name)
    print(pd.read_csv(f"{out}/log.csv").round(3).to_string(index=False))
    print()

## How to read Cell 4

- `human_eval_loss` is the loss on the held-out prompts' **human** answers, and `gemma_eval_loss` on their **Gemma** answers. Lower is better.
- Epoch 0 is the pretrained model before any fine-tuning, so it is the same for A and B.
- Model A is expected to do better on the human answers and model B on the Gemma answers, since each imitates its own training answers. The interesting question is the cross results and what the generated samples look like. Loss against one reference answer does not measure answer quality, so the blind comparison on held-out prompts is the main result.
- Checkpoints are saved to `tensorix/v2/sft/sft_human/sft.pt` and `.../sft_gemma/sft.pt`.